In [1]:
import polars as pl
import yaml

with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [2]:
DIR = f"{config['dirs']['output']}/{config['city']}_ids_not_replaced"

In [3]:
for i in range(1):
    temp = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet")

In [5]:
temp

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
"""0000051fc26276495c5a9ab1753a70…",1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,11
"""0000051fc26276495c5a9ab1753a70…",2,2024-10-02 10:23:47 CEST,2024-10-02 10:33:00 CEST,9m 13s,47.661216,19.076262,4.0,1.0,2,11
"""0000051fc26276495c5a9ab1753a70…",3,2024-10-03 10:50:40 CEST,2024-10-03 10:58:29 CEST,7m 49s,47.498402,19.0404,0.0,0.0,2,12
"""0000051fc26276495c5a9ab1753a70…",4,2024-10-04 12:41:20 CEST,2024-10-04 12:49:26 CEST,8m 6s,47.498402,19.0404,0.0,0.0,2,12
"""0000051fc26276495c5a9ab1753a70…",5,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,12
…,…,…,…,…,…,…,…,…,…,…
"""00ffd69f0786f191b711f219fd6605…",381,2024-11-17 18:27:21 CET,2024-11-17 18:37:33 CET,10m 12s,47.606706,19.270747,14.0,12.0,3,4
"""00ffd69f0786f191b711f219fd6605…",382,2024-11-17 21:11:36 CET,2024-11-17 21:26:20 CET,14m 44s,47.606829,19.270937,11.0,6.0,3,4
"""00ffd69f0786f191b711f219fd6605…",383,2024-11-18 04:47:22 CET,2024-11-18 05:13:19 CET,25m 57s,47.60688,19.270975,9.0,2.0,4,4


In [ ]:
import polars_distance as pld
# https://ion-elgreco.github.io/polars-distance/api/dist/#polars_distance.DistancePairWise.haversine

In [ ]:
data.select(
    pld.col("longitude_mean").dist.haversine("latitude_mean", "km").alias("haversine")
)

In [ ]:
data = temp.filter(pl.col.device_id.str.starts_with("0000051fc26276495c5a9ab1753a70"))[
    :3
].clone()

In [72]:
def rg(data: pl.DataFrame) -> pl.DataFrame:
    d = data.with_columns(
        pl.struct("longitude_mean", "latitude_mean")
        .alias("stop")
        .struct.rename_fields(["longitude", "latitude"]),
        pl.struct(pl.col.longitude_mean.mean(), pl.col.latitude_mean.mean())
        .alias("centroid")
        .struct.rename_fields(["longitude", "latitude"]),
    ).with_columns(
        pld.col("centroid").dist.haversine("stop", "km").alias("haversine"),
        pl.col.interval_lenght.dt.total_seconds().sum().alias("N"),
    )
    res = (
        ((d["haversine"] ** 2 * d["interval_lenght"].dt.total_seconds()).sum() / d["N"])
        .sqrt()
        .round(3)[0]
    )
    return data.head(1).with_columns(pl.lit(res).alias("result"))


In [58]:
q = rg(data)
q

8.241283721822473

In [50]:
((q["haversine"] ** 2 * q["interval_lenght"].dt.total_seconds()).sum() / q["N"]).sqrt()

N
f64
8.241284
8.241284
8.241284


In [74]:
temp.group_by("device_id").map_groups(rg)


device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,result
str,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,f64
"""00f99a3c6cd32fff46b762f5887d06…",1,2024-12-08 17:56:03 CET,2024-12-08 18:01:17 CET,5m 14s,47.513246,19.057419,7.0,3.0,6,1,5.311
"""00e319ee8d96cc0dbdba66f27707a3…",1,2025-08-16 12:49:08 CEST,2025-08-16 12:54:35 CEST,5m 27s,47.493563,19.049912,5.0,6.0,36,2,2.738
"""00347ff2dc3587d0df158fa14f9a3c…",1,2025-04-30 06:24:59 CEST,2025-04-30 06:49:06 CEST,24m 7s,47.19,18.65,0.0,0.0,7,0,65.05
"""00a78669ba2d88b092e4476a294ce1…",1,2025-02-14 01:11:13 CET,2025-02-14 01:21:40 CET,10m 27s,47.464585,19.120769,2.0,0.0,2,2,6.96
"""0005830e04f56ea36c12449f4f36b0…",1,2025-08-30 02:11:03 CEST,2025-08-30 05:48:39 CEST,3h 37m 36s,47.581395,18.386533,1.0,0.0,20,1,11.185
…,…,…,…,…,…,…,…,…,…,…,…
"""002c3d7cb948209a7b8f64b0c7378c…",1,2024-12-06 09:21:38 CET,2024-12-06 09:28:52 CET,7m 14s,47.541316,19.03555,0.0,0.0,2,3,0.119
"""00e45daa0c2a0fa7123822319c782a…",1,2025-10-30 20:57:48 CET,2025-10-30 21:08:19 CET,10m 31s,47.521766,19.179941,2.0,3.0,3,3,3.231
"""00f0ab95ffccce64359b6740733d97…",1,2025-12-04 15:16:58 CET,2025-12-04 15:27:31 CET,10m 33s,47.487447,19.072624,0.0,0.0,2,5,2.578


In [ ]:
temp.group_by(pl.col.device_id).with_columns(pl.col.longitude_mean)

In [ ]:
def calculate_radius_of_gyration(data, weighted: bool = False):
    rcm_lon = data["lon"].mean()
    rcm_lat = data["lat"].mean()
    freq = data.groupby(["x", "y"])["t"].count().to_dict()
    ds = []
    w = 0
    for row in data.itertuples():
        w += freq[(row.x, row.y)]
        d = haversine((rcm_lat, rcm_lon), (row.lat, row.lon), unit=Unit.METERS) ** 2
        if weighted:
            d *= freq[(row.x, row.y)]
        ds.append(d)

    sum_ = np.sum(ds)
    N = len(data)
    if weighted:
        N = w
    return np.round(np.sqrt(sum_ / N) / 1000, 3)